# Banking loan default: logistic regression and coefficients (weights)

**Goal:** predict `defaulted` (`1` = default, `0` = no default) from four applicant features. This is a **synthetic teaching dataset**, not real customer data. All four notebooks use the same CSV, feature order, random seed, train/test split, and scaling so model structures can be compared fairly.

Keep `Banking_Loan_Default_2400.csv` in the same folder as this notebook. Run the cells in order with Windows with Python 3.13.5. The final metric values depend on installed library versions and stochastic training.

## Windows setup (Python 3.13.5)
In Windows Command Prompt or PowerShell, run:
```powershell
py -3.13 -m pip install --upgrade pip
py -3.13 -m pip install notebook pandas numpy scikit-learn matplotlib tensorflow
```
Start Jupyter with `py -3.13 -m notebook`. Select the Python 3.13 kernel. TensorFlow includes the `tf.keras` API used in the ANN notebooks. A regular Windows CPU is sufficient for this small exercise.

## Step 1 - Import libraries
`pandas` reads the CSV; NumPy handles arrays; scikit-learn splits, scales, and evaluates. The ANN notebooks additionally use `tf.keras`, the Keras API included with TensorFlow.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

## Step 2 - Read and inspect the banking CSV
Each row represents one fictional applicant. `defaulted` is the answer to predict, so it must never be included among the input features.

In [ ]:
df = pd.read_csv("Banking_Loan_Default_2400.csv")
print("Rows and columns:", df.shape)
display(df.head())
print("Missing values:\n", df.isna().sum())
print("Default counts:\n", df["defaulted"].value_counts())

## Step 3 - Choose inputs and split the data
`X` contains exactly four input features; `y` contains the target. `test_size=0.20` reserves 20% for final evaluation. `stratify=y` keeps approximately the same default rate in train and test. `random_state=42` makes the split repeatable.

In [ ]:
features = ["age_years", "annual_income_usd", "outstanding_debt_usd", "credit_score"]
X = df[features]
y = df["defaulted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))

## Step 4 - Standardize inputs without leaking test information
A credit score and income have very different numeric ranges. `StandardScaler` puts them on comparable scales. Fit it on training data only, then use that fitted scaler on the test data. A standardized value means the number of training standard deviations above or below the training mean.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(pd.DataFrame(X_train_scaled[:3], columns=features).round(2))

## Step 5 - Train logistic regression
The model learns one coefficient (weight) per feature and one intercept (bias). Its score is `z = w1*x1 + w2*x2 + w3*x3 + w4*x4 + b`; sigmoid turns that score into a probability. `max_iter=1000` gives the solver enough iterations. Regularization is enabled by default.

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

## Step 6 - Inspect learned weights and bias
`coef_` contains four learned weights in the same order as `features`; `intercept_` contains the bias `b`. Because inputs were standardized, coefficients are comparable per one standard deviation change, holding other features fixed. A positive weight increases the log-odds of default; a negative weight decreases them. A coefficient is **not** an error.

In [ ]:
weights = pd.DataFrame({"feature": features, "weight": model.coef_[0]})
weights["odds_multiplier_per_1_SD"] = np.exp(weights["weight"])
display(weights.round(3))
print("Bias (intercept b):", round(float(model.intercept_[0]), 4))

## Step 7 - Calculate one applicant prediction by hand
Use the already scaled first test row. Multiply each scaled input by its learned weight; add all four contributions and the bias to get `z`. Then calculate `sigmoid(z) = 1 / (1 + exp(-z))`. The final line checks the manual probability against `predict_proba`.

In [ ]:
example = X_test_scaled[0]
contributions = example * model.coef_[0]
manual_z = contributions.sum() + model.intercept_[0]
manual_probability = 1 / (1 + np.exp(-manual_z))
display(pd.DataFrame({"feature": features, "scaled_input": example, "weight": model.coef_[0], "input_times_weight": contributions}).round(3))
print("Sum of contributions:", round(float(contributions.sum()), 4))
print("Bias b:", round(float(model.intercept_[0]), 4))
print("z:", round(float(manual_z), 4))
print("Manual probability:", round(float(manual_probability), 4))
print("Model probability:", round(float(model.predict_proba(X_test_scaled)[0, 1]), 4))

## Step 8 - Evaluate on unseen applicants
`accuracy` is the fraction of correct classifications. Precision, recall, and F1 add information about the default class. ROC AUC measures how well probabilities rank positive versus negative cases. Use `0.5` only as a teaching threshold; a real bank would select a threshold based on costs and policy.

In [ ]:
probabilities = model.predict_proba(X_test_scaled)[:, 1]
predictions = (probabilities >= 0.5).astype(int)
print("Accuracy:", round(accuracy_score(y_test, predictions), 3))
print("ROC AUC:", round(roc_auc_score(y_test, probabilities), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, predictions))
print(classification_report(y_test, predictions, target_names=["No default", "Default"]))

## Step 9 - Interpret coefficients visually
The horizontal axis is the learned weight for a one-standard-deviation change. The sign gives the direction of association while other features remain fixed. Correlation among inputs means coefficients should not be treated as causal effects.

In [ ]:
weights.sort_values("weight").plot.barh(x="feature", y="weight", legend=False, figsize=(8, 4), color="#2563eb")
plt.axvline(0, color="black", linewidth=1)
plt.title("Logistic regression weights on standardized inputs")
plt.xlabel("Weight (change in log-odds per 1 standard deviation)")
plt.tight_layout()
plt.show()

## Step 10 - Connect to the neural network notebooks
Logistic regression combines the four inputs directly into one output. The next notebook inserts a hidden layer between them. Each hidden neuron also learns weights and a bias, computes a weighted sum, and applies an activation. More layers do not guarantee better test results.